In [3]:
import shutil
from pathlib import Path

from src.schemas import dispatch
from src.schemas.parsers.pdf_extractor import PDFExtractor
from src.utils.helpers import totalizador, planilha_lancamento

from src.app.gdrive.google_drive_auth import GoogleDriveAuth
from src.app.gdrive.settings import (
    GOOGLE_OAUTH_CREDENTIALS,
    GOOGLE_OAUTH_TOKEN,
    GOOGLE_DRIVE_FOLDER_ID,
    PDF_MIME_TYPE,
    XLSX_MIME_TYPE,
    XLS_MIME_TYPE,
)

def main():
    google_drive = GoogleDriveAuth(
        credentials_path=GOOGLE_OAUTH_CREDENTIALS,
        token_path=GOOGLE_OAUTH_TOKEN
    )
    # ID da pasta AUTOEXT no Google Drive
    extratos_id = GOOGLE_DRIVE_FOLDER_ID

    # Arquivo modelo local
    lancamento = Path.cwd() / "data" / "Lancamentos_Contabeis.xls"

    # Pasta temporária local
    temp_dir = Path.cwd() / "temp"
    temp_dir.mkdir(parents=True, exist_ok=True)

    # Cria ou recupera as pastas principais no Google Drive
    pasta_invalidos = google_drive.get_or_create_folder(
        folder_id_pai=extratos_id,
        name_folder="00_INVALIDOS"
    )

    pasta_convertidos = google_drive.get_or_create_folder(
        folder_id_pai=extratos_id,
        name_folder="00_CONVERTIDOS"
    )

    invalidos_id = pasta_invalidos["id"]
    convertidos_id = pasta_convertidos["id"]

    # Lista PDFs da pasta AUTOEXT
    arquivos = google_drive.pdfs(
        folder_id=extratos_id,
        pdf_type=PDF_MIME_TYPE
    )
    
    if not arquivos:
        print("Pasta vazia")

    for arquivo_drive in arquivos:
        arquivo_id = arquivo_drive["id"]
        arquivo_nome = arquivo_drive["name"]

        arquivo_path = Path(arquivo_nome)
        arquivo_stem = arquivo_path.stem

        if "EXT" not in arquivo_stem:
            continue

        print(f"\nProcessando PDF: {arquivo_nome}")

        # Caminho local temporário do PDF baixado
        pdf_local = temp_dir / arquivo_nome

        # Baixa o PDF do Google Drive para processar localmente
        google_drive.download(
            file_id=arquivo_id,
            destino_local=pdf_local
        )

        pdf = PDFExtractor(pdf_local).extract()

        if not pdf:
            google_drive.move_file(
                file_id=arquivo_id,
                folder_id_destino=invalidos_id
            )

            pdf_local.unlink(missing_ok=True)

            print(f"PDF movido para inválidos porque está vazio: {arquivo_nome}")
            continue

        df = dispatch(pdf)

        if df is None or df.empty:
            google_drive.move_file(
                file_id=arquivo_id,
                folder_id_destino=invalidos_id
            )

            pdf_local.unlink(missing_ok=True)

            print(f"PDF movido para inválidos porque o DataFrame veio vazio: {arquivo_nome}")
            continue

        # Cria ou recupera uma pasta dentro de 00_CONVERTIDOS com o nome do arquivo
        pasta_arquivo = google_drive.get_or_create_folder(
            folder_id_pai=convertidos_id,
            name_folder=arquivo_stem
        )

        pasta_arquivo_id = pasta_arquivo["id"]

        # Nome do lançamento
        nome_lancamento = arquivo_stem.replace("EXT", "LANC")

        # Arquivos locais temporários gerados
        dest_lancamento = temp_dir / f"{nome_lancamento}.xls"
        dest_excel = temp_dir / f"{arquivo_stem}.xlsx"

        # Copia o modelo de lançamento para a pasta temporária
        shutil.copy2(lancamento, dest_lancamento)

        # Preenche a planilha de lançamento
        planilha_lancamento(df, dest_lancamento)

        # Gera o Excel totalizado
        df_totalizado = totalizador(df)
        df_totalizado.to_excel(dest_excel, index=False)

        # Upload do XLS para o Google Drive
        google_drive.upload(
            caminho_local=dest_lancamento,
            folder_id_destino=pasta_arquivo_id,
            type_file=XLS_MIME_TYPE,
            name_drive=dest_lancamento.name
        )

        # Upload do XLSX para o Google Drive
        google_drive.upload(
            caminho_local=dest_excel,
            folder_id_destino=pasta_arquivo_id,
            type_file=XLSX_MIME_TYPE,
            name_drive=dest_excel.name
        )

        # Move o PDF original no Google Drive para a pasta do arquivo convertido
        google_drive.move_file(
            file_id=arquivo_id,
            folder_id_destino=pasta_arquivo_id
        )

        # Limpa arquivos temporários locais
        pdf_local.unlink(missing_ok=True)
        dest_lancamento.unlink(missing_ok=True)
        dest_excel.unlink(missing_ok=True)

        print(f"PDF convertido com sucesso: {arquivo_nome}")
        print(f"Arquivos enviados para a pasta: {arquivo_stem}")

if __name__ == "__main__":
    main()

Pasta vazia
